<a href="https://colab.research.google.com/github/kyle29-git/kyle29-git_64061/blob/main/Assignment_3_Time_Series_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Insert Data

import os
import numpy as np
import keras
from keras import layers

fname = "jena_climate_2009_2016.csv"

with open(fname) as f:
    data = f.read()

lines = data.split("\n")
header = lines[0].split(",")
lines = lines[1:]

clean_lines = [line for line in lines if line.strip() != ""]

temperature = []
raw_data = []

for line in clean_lines:
    values = line.split(",")

    if len(values) != len(header):
        continue

    values = [float(x) for x in values[1:]]

    temperature.append(values[1])
    raw_data.append(values)

temperature = np.array(temperature)
raw_data = np.array(raw_data)

num_train = int(0.5 * len(raw_data))
num_val = int(0.25 * len(raw_data))

mean = raw_data[:num_train].mean(axis=0)
raw_data -= mean

std = raw_data[:num_train].std(axis=0)
raw_data /= std

In [6]:
# Creating Time Series Datasets

sampling_rate = 6
sequence_length = 120
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train,
)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=num_train,
    end_index=num_train + num_val,
)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=num_train + num_val,
)

In [7]:
# Model 1
# Baseline LSTM

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(32)(inputs)
outputs = layers.Dense(1)(x)

model1 = keras.Model(inputs, outputs)
model1.compile(optimizer="adam", loss="mse", metrics=["mae"])

history1 = model1.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

Epoch 1/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 131s 157ms/step - loss: 40.5898 - mae: 4.7681 - val_loss: 20.3510 - val_mae: 3.5546
Epoch 2/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 146s 179ms/step - loss: 14.5867 - mae: 2.9427 - val_loss: 14.5048 - val_mae: 2.9968
Epoch 3/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 136s 166ms/step - loss: 12.1851 - mae: 2.7281 - val_loss: 13.0077 - val_mae: 2.8437
Epoch 4/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 127s 156ms/step - loss: 11.6464 - mae: 2.6622 - val_loss: 12.9833 - val_mae: 2.8541
Epoch 5/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 126s 154ms/step - loss: 11.3914 - mae: 2.6361 - val_loss: 12.1533 - val_mae: 2.7423
Epoch 6/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 124s 151ms/step - loss: 11.0538 - mae: 2.6035 - val_loss: 11.3746 - val_mae: 2.6498
Epoch 7/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 129s 157ms/step - loss: 10.8220 - mae: 2.5773 - val_loss: 11.4370 - val_mae: 2.6605
Epoch 8/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 123s 150ms/step - loss: 10.6782 - mae: 2.5595 - val_loss: 10.7401 - val_mae: 2.5704
Epoch 9/

In [8]:
# Model 2
# LSTM and Dropout
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

model2 = keras.Model(inputs, outputs)
model2.compile(optimizer="adam", loss="mse", metrics=["mae"])

history2 = model2.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset
)

Epoch 1/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 164s 196ms/step - loss: 46.7981 - mae: 5.1752 - val_loss: 20.7779 - val_mae: 3.5963
Epoch 2/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 159s 195ms/step - loss: 20.2798 - mae: 3.4765 - val_loss: 14.7153 - val_mae: 3.0113
Epoch 3/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 203s 196ms/step - loss: 17.7861 - mae: 3.2761 - val_loss: 13.0247 - val_mae: 2.8295
Epoch 4/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 165s 202ms/step - loss: 16.8205 - mae: 3.1907 - val_loss: 12.4012 - val_mae: 2.7607
Epoch 5/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 204s 204ms/step - loss: 16.2477 - mae: 3.1383 - val_loss: 11.9990 - val_mae: 2.7188
Epoch 6/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 169s 206ms/step - loss: 15.9496 - mae: 3.1108 - val_loss: 11.9009 - val_mae: 2.7034
Epoch 7/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 177s 216ms/step - loss: 15.7720 - mae: 3.0902 - val_loss: 11.5932 - val_mae: 2.6710
Epoch 8/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 176s 214ms/step - loss: 15.5898 - mae: 3.0688 - val_loss: 11.3135 - val_mae: 2.6351
Epoch 9/

In [9]:
# Model 3
# CNN and GRU
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Conv1D(32, 5, activation="relu")(inputs)
x = layers.MaxPooling1D(3)(x)
x = layers.GRU(32)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)

model3 = keras.Model(inputs, outputs)
model3.compile(optimizer="adam", loss="mse", metrics=["mae"])

history3 = model3.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset
)

Epoch 1/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 120s 142ms/step - loss: 39.7759 - mae: 4.8676 - val_loss: 24.6624 - val_mae: 3.8579
Epoch 2/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 113s 138ms/step - loss: 21.8745 - mae: 3.6381 - val_loss: 29.8305 - val_mae: 4.3187
Epoch 3/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 110s 134ms/step - loss: 19.6472 - mae: 3.4585 - val_loss: 24.3837 - val_mae: 3.9364
Epoch 4/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 111s 136ms/step - loss: 18.3374 - mae: 3.3425 - val_loss: 25.3519 - val_mae: 4.0213
Epoch 5/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 109s 133ms/step - loss: 17.5156 - mae: 3.2676 - val_loss: 24.7245 - val_mae: 3.9796
Epoch 6/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 109s 133ms/step - loss: 17.2307 - mae: 3.2408 - val_loss: 24.8574 - val_mae: 3.9907
Epoch 7/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 112s 137ms/step - loss: 16.8083 - mae: 3.1963 - val_loss: 22.6794 - val_mae: 3.8159
Epoch 8/20
819/819 ━━━━━━━━━━━━━━━━━━━━ 108s 131ms/step - loss: 16.4974 - mae: 3.1651 - val_loss: 23.8740 - val_mae: 3.9097
Epoch 9/

In [10]:
# Comparison

print("Model 1 Test MAE:", model1.evaluate(test_dataset)[1])
print("Model 2 Test MAE:", model2.evaluate(test_dataset)[1])
print("Model 3 Test MAE:", model3.evaluate(test_dataset)[1])

results = {
    "Model": ["LSTM", "LSTM + Dropout", "CNN + GRU"],
    "Val MAE": [
        min(history1.history["val_mae"]),
        min(history2.history["val_mae"]),
        min(history3.history["val_mae"])
    ]
}

import pandas as pd
df = pd.DataFrame(results)
print(df)

405/405 ━━━━━━━━━━━━━━━━━━━━ 31s 76ms/step - loss: 12.3257 - mae: 2.7342
Model 1 Test MAE: 2.7341670989990234
405/405 ━━━━━━━━━━━━━━━━━━━━ 31s 76ms/step - loss: 12.0629 - mae: 2.7104
Model 2 Test MAE: 2.71040678024292
405/405 ━━━━━━━━━━━━━━━━━━━━ 25s 63ms/step - loss: 18.9733 - mae: 3.4388
Model 3 Test MAE: 3.438767671585083
            Model   Val MAE
0            LSTM  2.570424
1  LSTM + Dropout  2.517063
2       CNN + GRU  3.327404
